# Personal Finance & Spending Analysis
## 03 — Exploratory Data Analysis (EDA)

**Project stage:** Business analysis and insight generation  
**Input:** `data/cleaned_data.csv`  
**Outputs:** `reports/eda_summary.csv`, `reports/business_insights.csv`, and charts in `visualizations/`

## Project objective
Analyze income, household spending, savings goals, and potential savings to identify financial patterns and actionable opportunities.

## Business questions
1. What does the income distribution look like?
2. Which expense categories account for the largest spending?
3. How does income relate to disposable income and desired savings?
4. Does age relate to spending and savings patterns?
5. How do income and expenses differ by occupation?
6. How do financial patterns differ across city tiers?
7. Which categories show the greatest potential savings?
8. How closely does disposable income align with an expense-based estimate?

## Analytical approach
- Use descriptive statistics and grouped comparisons.
- Visualize distributions, category shares, and relationships.
- Distinguish observed values from estimates derived in this notebook.
- Avoid causal claims: patterns and correlations do not prove cause and effect.

> **Run instructions:** Save this notebook in `notebooks/03_eda_analysis.ipynb`. It expects the cleaned file at `data/cleaned_data.csv` in the project root. Run all cells from top to bottom.

## 1. Import Libraries and Configure Project Paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHARTS_DIR = PROJECT_ROOT / "visualizations"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths configured.")

## 2. Load and Validate the Cleaned Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {DATA_PATH.resolve()}\n"
        "Run 02_data_cleaning.ipynb first and confirm data/cleaned_data.csv exists."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset size: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())
df.info()

In [ ]:
# Check the core fields used throughout this analysis.
required_columns = ["income"]
missing_required = [column for column in required_columns if column not in df.columns]

if missing_required:
    raise ValueError(f"Required columns missing from cleaned dataset: {missing_required}")

print("Core analysis fields are available.")

## 3. Define Analysis Variables

In [ ]:
# Expense categories used in the supplied dataset.
expense_columns = [
    "rent", "loan_repayment", "insurance", "groceries", "transport",
    "eating_out", "entertainment", "utilities", "healthcare",
    "education", "miscellaneous"
]
expense_columns = [column for column in expense_columns if column in df.columns]

potential_savings_columns = [
    column for column in df.columns
    if column.startswith("potential_savings_")
]

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Expense columns:", expense_columns)
print("Potential savings columns:", potential_savings_columns)
print("Categorical columns:", categorical_columns)

In [ ]:
# Create analysis-only metrics. The source columns are not overwritten.
if expense_columns:
    df["analysis_total_expenses"] = df[expense_columns].sum(axis=1, min_count=1)
else:
    df["analysis_total_expenses"] = np.nan

if "income" in df.columns:
    df["analysis_expense_ratio_pct"] = (
        df["analysis_total_expenses"] / df["income"].replace(0, np.nan) * 100
    )

# An expense-based estimate; compare it with the provided disposable_income,
# but do not assume the source uses exactly this formula.
df["analysis_estimated_remaining_income"] = df["income"] - df["analysis_total_expenses"]

print("Analysis metrics created.")
display(df[["income", "analysis_total_expenses",
            "analysis_estimated_remaining_income"]].head())

## 4. Dataset Overview and Data Quality Snapshot

In [ ]:
overview = pd.DataFrame({
    "metric": [
        "Records", "Columns", "Missing cells",
        "Duplicate rows", "Expense categories available",
        "Potential-savings categories available"
    ],
    "value": [
        len(df), df.shape[1], int(df.isna().sum().sum()),
        int(df.duplicated().sum()), len(expense_columns),
        len(potential_savings_columns)
    ]
})
display(overview)

## 5. Business Question 1 — What Does the Income Distribution Look Like?

In [ ]:
income_summary = df["income"].describe().to_frame("income").T
display(income_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["income"].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title("Income Distribution")
axes[0].set_xlabel("Income")

sns.boxplot(x=df["income"], ax=axes[1])
axes[1].set_title("Income — Outlier Review")
axes[1].set_xlabel("Income")

plt.tight_layout()
plt.savefig(CHARTS_DIR / "01_income_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**How to interpret:** Compare mean and median. A mean much higher than the median may indicate a right-skewed distribution, where a smaller number of high-income records raise the average. Treat boxplot outliers as records for investigation, not automatically as errors.

## 6. Business Question 2 — Which Expense Categories Have the Highest Spending?

In [ ]:
if not expense_columns:
    print("No recognized expense columns were found.")
else:
    expense_summary = pd.DataFrame({
        "total_spending": df[expense_columns].sum(min_count=1),
        "average_spending_per_record": df[expense_columns].mean(),
        "median_spending_per_record": df[expense_columns].median()
    }).sort_values("total_spending", ascending=False)

    expense_summary["share_of_total_expenses_pct"] = (
        expense_summary["total_spending"] /
        expense_summary["total_spending"].sum() * 100
    )
    display(expense_summary)

    ax = expense_summary["total_spending"].sort_values().plot(
        kind="barh", figsize=(10, 6), title="Total Spending by Expense Category"
    )
    ax.set_xlabel("Total spending")
    ax.set_ylabel("Expense category")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "02_expense_category_totals.png", dpi=150, bbox_inches="tight")
    plt.show()

**How to interpret:** Use total spending to understand the overall contribution of each category, and average/median spending to understand a typical record. A category with a high total may be high because it is large per household, common across households, or both.

## 7. Business Question 3 — How Does Income Relate to Disposable Income and Desired Savings?

In [ ]:
relationship_columns = [
    column for column in [
        "income", "disposable_income", "desired_savings",
        "desired_savings_percentage", "analysis_total_expenses"
    ] if column in df.columns
]

display(df[relationship_columns].corr(numeric_only=True))

if "disposable_income" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.regplot(
        data=df, x="income", y="disposable_income",
        scatter_kws={"alpha": 0.35}, line_kws={"color": "red"}, ax=ax
    )
    ax.set_title("Income vs. Disposable Income")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "03_income_vs_disposable_income.png", dpi=150, bbox_inches="tight")
    plt.show()

if "desired_savings" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=df, x="income", y="desired_savings", alpha=0.45, ax=ax)
    ax.set_title("Income vs. Desired Savings")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "04_income_vs_desired_savings.png", dpi=150, bbox_inches="tight")
    plt.show()

**How to interpret:** Review the direction and strength of correlations, and inspect the scatterplots for spread and unusual observations. Correlation does not establish that higher income causes higher savings.

## 8. Business Question 4 — Does Age Relate to Spending and Savings Patterns?

In [ ]:
if "age" not in df.columns:
    print("Age column not available; age analysis skipped.")
else:
    age_bins = [0, 24, 34, 44, 54, 64, 120]
    age_labels = ["18–24", "25–34", "35–44", "45–54", "55–64", "65+"]

    df["analysis_age_group"] = pd.cut(
        df["age"], bins=age_bins, labels=age_labels, include_lowest=True
    )

    age_metrics = [
        column for column in [
            "income", "analysis_total_expenses", "disposable_income",
            "desired_savings", "dependents"
        ] if column in df.columns
    ]

    age_summary = df.groupby("analysis_age_group", observed=False)[age_metrics].agg(
        ["count", "mean", "median"]
    )
    display(age_summary)

    if "analysis_total_expenses" in age_metrics:
        age_plot = df.groupby("analysis_age_group", observed=False)[
            "analysis_total_expenses"
        ].mean()
        ax = age_plot.plot(kind="bar", figsize=(9, 5), title="Average Expenses by Age Group")
        ax.set_xlabel("Age group")
        ax.set_ylabel("Average expenses")
        plt.tight_layout()
        plt.savefig(CHARTS_DIR / "05_average_expenses_by_age_group.png", dpi=150, bbox_inches="tight")
        plt.show()

**How to interpret:** Compare group sizes as well as averages. Some age bands may have fewer records, so their averages may be less stable. Age groups are descriptive segments, not evidence of age causing spending differences.

## 9. Business Question 5 — How Do Income and Expenses Differ by Occupation?

In [ ]:
if "occupation" not in df.columns:
    print("Occupation column not available; occupation analysis skipped.")
else:
    occupation_metrics = [
        column for column in [
            "income", "analysis_total_expenses", "disposable_income",
            "desired_savings"
        ] if column in df.columns
    ]

    occupation_summary = df.groupby("occupation", dropna=False)[occupation_metrics].agg(
        ["count", "mean", "median"]
    )
    display(occupation_summary)

    occupation_income = df.groupby("occupation", dropna=False)["income"].median().sort_values()
    ax = occupation_income.plot(
        kind="barh", figsize=(9, 5), title="Median Income by Occupation"
    )
    ax.set_xlabel("Median income")
    ax.set_ylabel("Occupation")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "06_median_income_by_occupation.png", dpi=150, bbox_inches="tight")
    plt.show()

**How to interpret:** Median income is less sensitive to extreme values than mean income. Check the number of records in each occupation before drawing conclusions.

## 10. Business Question 6 — How Do Financial Patterns Differ Across City Tiers?

In [ ]:
if "city_tier" not in df.columns:
    print("City_Tier column not available; city-tier analysis skipped.")
else:
    city_metrics = [
        column for column in [
            "income", "analysis_total_expenses", "rent", "groceries",
            "transport", "disposable_income"
        ] if column in df.columns
    ]

    city_summary = df.groupby("city_tier", dropna=False)[city_metrics].agg(
        ["count", "mean", "median"]
    )
    display(city_summary)

    if "rent" in df.columns:
        rent_by_city = df.groupby("city_tier", dropna=False)["rent"].median().sort_values()
        ax = rent_by_city.plot(kind="bar", figsize=(8, 5), title="Median Rent by City Tier")
        ax.set_xlabel("City tier")
        ax.set_ylabel("Median rent")
        plt.tight_layout()
        plt.savefig(CHARTS_DIR / "07_median_rent_by_city_tier.png", dpi=150, bbox_inches="tight")
        plt.show()

**How to interpret:** Compare rent and total expenses alongside income. A higher expense amount does not necessarily mean greater financial pressure if income is also higher.

## 11. Business Question 7 — Which Categories Have the Greatest Potential Savings?

In [ ]:
if not potential_savings_columns:
    print("Potential savings columns not found; this analysis is skipped.")
else:
    potential_summary = pd.DataFrame({
        "total_potential_savings": df[potential_savings_columns].sum(min_count=1),
        "average_potential_savings": df[potential_savings_columns].mean(),
        "median_potential_savings": df[potential_savings_columns].median()
    }).sort_values("total_potential_savings", ascending=False)

    potential_summary.index = (
        potential_summary.index.str.replace("potential_savings_", "", regex=False)
    )
    display(potential_summary)

    ax = potential_summary["total_potential_savings"].sort_values().plot(
        kind="barh", figsize=(10, 6), title="Potential Savings by Category"
    )
    ax.set_xlabel("Potential savings")
    ax.set_ylabel("Expense category")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "08_potential_savings_by_category.png", dpi=150, bbox_inches="tight")
    plt.show()

**How to interpret:** These are dataset-provided potential-savings fields. They represent modeled opportunities, not guaranteed savings. Confirm how the source dataset calculated them before making recommendations.

## 12. Business Question 8 — Does Disposable Income Align with an Expense-Based Estimate?

In [ ]:
if "disposable_income" not in df.columns:
    print("Provided disposable_income column not available; reconciliation skipped.")
else:
    reconciliation = df[[
        "income", "analysis_total_expenses",
        "analysis_estimated_remaining_income", "disposable_income"
    ]].copy()

    reconciliation["difference_vs_provided_disposable"] = (
        reconciliation["analysis_estimated_remaining_income"] -
        reconciliation["disposable_income"]
    )

    display(reconciliation.describe().T)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(
        reconciliation["difference_vs_provided_disposable"].dropna(),
        bins=30, kde=True, ax=ax
    )
    ax.axvline(0, linestyle="--", color="red")
    ax.set_title("Estimated Remaining Income minus Provided Disposable Income")
    ax.set_xlabel("Difference")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "09_disposable_income_reconciliation.png", dpi=150, bbox_inches="tight")
    plt.show()

**Important interpretation:** `analysis_estimated_remaining_income` is calculated here as income minus the listed expense categories. The source `disposable_income` may use a different formula or include other assumptions. A non-zero difference is a reconciliation question, not automatically a data error.

## 13. Key Findings — Automatically Generated Evidence

In [ ]:
# Generate concise descriptive findings from the available data.
findings = []

findings.append({
    "business_question": "Dataset overview",
    "finding": f"The dataset contains {len(df):,} records and {df.shape[1]:,} columns."
})

if df["income"].notna().any():
    findings.append({
        "business_question": "Income distribution",
        "finding": (
            f"Median income is {df['income'].median():,.2f}; "
            f"mean income is {df['income'].mean():,.2f}."
        )
    })

if expense_columns:
    expense_totals = df[expense_columns].sum(min_count=1).dropna()
    if not expense_totals.empty:
        top_expense = expense_totals.idxmax()
        findings.append({
            "business_question": "Highest total expense category",
            "finding": f"{top_expense} has the highest total spending ({expense_totals.max():,.2f})."
        })

if "occupation" in df.columns:
    occ_median = df.groupby("occupation")["income"].median().dropna()
    if not occ_median.empty:
        findings.append({
            "business_question": "Occupation comparison",
            "finding": (
                f"{occ_median.idxmax()} has the highest median income "
                f"among occupation groups ({occ_median.max():,.2f})."
            )
        })

if "city_tier" in df.columns and "rent" in df.columns:
    city_rent = df.groupby("city_tier")["rent"].median().dropna()
    if not city_rent.empty:
        findings.append({
            "business_question": "City-tier rent comparison",
            "finding": (
                f"{city_rent.idxmax()} has the highest median rent "
                f"({city_rent.max():,.2f})."
            )
        })

if potential_savings_columns:
    pot_totals = df[potential_savings_columns].sum(min_count=1).dropna()
    if not pot_totals.empty:
        top_potential = pot_totals.idxmax().replace("potential_savings_", "")
        findings.append({
            "business_question": "Largest modeled savings opportunity",
            "finding": (
                f"{top_potential} has the highest total potential savings "
                f"({pot_totals.max():,.2f})."
            )
        })

findings_df = pd.DataFrame(findings)
display(findings_df)

## 14. Export EDA Results

In [ ]:
# Export a compact numerical summary for portfolio reproducibility.
summary_rows = []

for column in ["income", "disposable_income", "desired_savings",
               "desired_savings_percentage", "analysis_total_expenses",
               "analysis_estimated_remaining_income"]:
    if column in df.columns:
        summary_rows.append({
            "metric": column,
            "count": int(df[column].count()),
            "mean": df[column].mean(),
            "median": df[column].median(),
            "min": df[column].min(),
            "max": df[column].max()
        })

eda_summary = pd.DataFrame(summary_rows)
eda_summary.to_csv(REPORTS_DIR / "eda_summary.csv", index=False)
findings_df.to_csv(REPORTS_DIR / "business_insights.csv", index=False)

print(f"Saved: {(REPORTS_DIR / 'eda_summary.csv').resolve()}")
print(f"Saved: {(REPORTS_DIR / 'business_insights.csv').resolve()}")
print(f"Charts saved in: {CHARTS_DIR.resolve()}")